# 01 — Grid & room check

Sanity-check the foundation (`config`, `grid`, `utils`) and the `Room` builder, rendered with the project's **own pygame renderer** (no matplotlib).

In [ ]:
import sys, pathlib, os
SRC = pathlib.Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless: render to images, no window

import numpy as np
import pygame
from IPython.display import Image, display

from config import MATERIALS
from grid import RoomGrid
from utils import coord_to_cell, cell_to_coord, empty_field, pressure_reflection
from render import field_to_rgb, to_surface
import scenes

pygame.init()

def show(alpha, field=None, p_scale=1.0, scale=5, fname="_render.png"):
    """Render an (NY, NX) alpha-map (+ optional pressure field) with the project's
    own pygame renderer and display it inline -- no matplotlib."""
    if field is None:
        field = np.zeros_like(alpha)
    surf = to_surface(field_to_rgb(field, alpha, p_scale))
    surf = pygame.transform.scale(surf, (alpha.shape[1] * scale, alpha.shape[0] * scale))
    pygame.image.save(surf, fname)
    display(Image(filename=fname))

## Grid parameters

`dx`, `dt`, the grid size and the Courant number are all derived from `config.py` — nothing is hand-tuned.

In [ ]:
grid = RoomGrid()
print(grid)

## Coordinate round-trip

`coord_to_cell` rounds to the nearest cell and clamps to the grid.

In [ ]:
for x, y in [(0, 0), (4, 3), (8, 6), (10, -1)]:
    r, c = coord_to_cell(x, y, grid)
    xb, yb = cell_to_coord(r, c, grid)
    print(f"({x}, {y}) m  ->  (row={r}, col={c})  ->  ({xb:.3f}, {yb:.3f}) m")

## Materials → reflection

The solver turns each absorption coefficient α into a pressure reflection `R_p = sqrt(1 - alpha)` (absorbed energy = `1 - R_p**2 = alpha`).

In [ ]:
for name, a in MATERIALS.items():
    print(f"{name:>14}  alpha={a:.2f}  ->  R_p={float(pressure_reflection(a)):.3f}")

## The room

A `Room` stamps an `(NY, NX)` α-map. Default **two-room** scene (white = air, grey = walls/furniture):

In [ ]:
room = scenes.two_rooms(grid)
print(room.summary())
show(room.alpha)

## A floor plan

The `tiny_home` preset — a concrete shell with windows, drywall partitions making several rooms, and furniture:

In [ ]:
home = scenes.tiny_home(grid)
print(home.summary())
show(home.alpha)